# Part A: Setup and Validate Data

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


# Load Data


In [8]:
DATA_PATH = Path("data/machines_ml.csv")
machines = pd.read_csv(DATA_PATH)

# Validate dataset

In [9]:
required = {"temp_c", "vibration", "runtime", "pressure", "fault"}
missing = required - set(machines.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
if machines[list(required)].isna().any().any():
    raise ValueError("Required columns contain missing values.")
if not set(machines["fault"].unique()).issubset({0, 1}):
    raise ValueError("fault must contain only 0 and 1.")

print("Data Shape:", machines.shape)
print("Target Class Proportions:\n", machines["fault"].value_counts(normalize=True))

Data Shape: (160, 6)
Target Class Proportions:
 fault
0    0.7
1    0.3
Name: proportion, dtype: float64


# Part B: Features and Split

In [11]:
FEATURES = ["temp_c", "vibration", "runtime", "pressure"]
X = machines[FEATURES]
y = machines["fault"]

# Create a reproducible, stratified split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")

Train rows: 128, Test rows: 32


# Part C: Train and Evaluate a Classifier

# C1. Train a baseline tree

In [12]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

train_accuracy = accuracy_score(y_train, train_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Baseline Tree - Train Accuracy: {train_accuracy:.4f}")
print(f"Baseline Tree - Test Accuracy: {test_accuracy:.4f}")

Baseline Tree - Train Accuracy: 0.8984
Baseline Tree - Test Accuracy: 0.7500


# C2. Interpret the result (Confusion Matrix)


In [13]:
cm = confusion_matrix(y_test, test_predictions)
tn, fp, fn, tp = cm.ravel()
manual_accuracy = (tn + tp) / cm.sum()

print("\nConfusion Matrix Results:")
print({"TN": tn, "FP": fp, "FN": fn, "TP": tp})
assert abs(manual_accuracy - test_accuracy) < 1e-12


Confusion Matrix Results:
{'TN': np.int64(19), 'FP': np.int64(3), 'FN': np.int64(5), 'TP': np.int64(5)}
